# Optimisation de la compacité géométrique — Assemblage glouton de polyominos

Implémentation Python de l'article :
  "Optimisation discrète de la compacité géométrique :
   Une double approche heuristique pour le pavage 2D"

Trois heuristiques sont implémentées et comparées :
  H1 — Ordre maximal (glouton rapide, O(V))
  H2 — Distance-produit d'ordres (adaptatif, O(V²))
  H3 — Force brute gloutonne (référentiel absolu)

Chaque heuristique est exécutée via un beam search multi-branches
afin d'éviter les optima locaux.

Deux métriques de ressemblance permettent de comparer morphologiquement
les solutions obtenues par H1 et H2 vis-à-vis de H3 :
  R_simple   = N_diff / A_tot
  R_pondérée = Σ w_i / A_tot   (avec w_i = degré d'excentricité)

## Utilisation

  ### Lancement avec les paramètres par défaut
  python compacite_gloutonne.py

  ### Depuis un notebook (Kaggle / Colab / Jupyter)
  main(n_shapes=50, seed=7)

Auteur  : [Garrido Andersson Karl]
Date    : 04/2026

# ALGORITHME

## BIBLIOTHEQUE

In [ ]:
from __future__ import annotations

import random
from dataclasses import dataclass
from typing import Optional

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba

## TYPE ALIAS

In [ ]:
Cell = tuple[int, int]

## STRUCTURE DE BASE

In [ ]:

@dataclass
class Shape:
    """
    Représente un bloc (polyomino) : ensemble connexe d'unités primaires.

    Attributs
    ---------
    id    : identifiant unique du bloc
    cells : ensemble (frozenset) des cellules (x, y) composant le bloc
    color : couleur de visualisation (hex string)

    Propriétés calculées
    --------------------
    area        : nombre de cellules (aire)
    perimeter   : nombre d'arêtes exposées (périmètre discret)
    compactness : c(f) = aire / périmètre  (Définition 3 de l'article)
    """

    id: int
    cells: frozenset[Cell]
    color: str = "#4ECDC4"

    def __post_init__(self) -> None:
        self.cells = frozenset(self.cells)

    # ── Métriques ─────────────────────────────────────────────────────────

    @property
    def area(self) -> int:
        return len(self.cells)

    @property
    def perimeter(self) -> int:
        p = 0
        for x, y in self.cells:
            for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
                if (x + dx, y + dy) not in self.cells:
                    p += 1
        return p

    @property
    def compactness(self) -> float:
        """c(f) = A / P  (Définition 3). Retourne 0 si le périmètre est nul."""
        p = self.perimeter
        return self.area / p if p > 0 else 0.0

    # ── Transformations géométriques ──────────────────────────────────────

    def normalized(self) -> Shape:
        """Translate pour que min(x) = 0 et min(y) = 0."""
        if not self.cells:
            return self
        min_x = min(x for x, _ in self.cells)
        min_y = min(y for _, y in self.cells)
        return Shape(
            self.id,
            frozenset((x - min_x, y - min_y) for x, y in self.cells),
            self.color,
        )

    def translated(self, dx: int, dy: int) -> Shape:
        """Retourne une copie translatée de (dx, dy)."""
        return Shape(
            self.id,
            frozenset((x + dx, y + dy) for x, y in self.cells),
            self.color,
        )

    def rotated(self, n: int) -> Shape:
        """Retourne une copie après n rotations de 90° dans le sens horaire, normalisée."""
        cells = list(self.cells)
        for _ in range(n % 4):
            cells = [(y, -x) for x, y in cells]
        return Shape(self.id, frozenset(cells), self.color).normalized()

    # ── Opérations ensemblistes ───────────────────────────────────────────

    def overlaps(self, other: Shape) -> bool:
        """True si les deux blocs partagent au moins une cellule."""
        return bool(self.cells & other.cells)

    def merged_with(self, other: Shape) -> Shape:
        """Retourne l'union des deux blocs (même id et couleur que self)."""
        return Shape(self.id, self.cells | other.cells, self.color)

    def __repr__(self) -> str:
        return (
            f"Shape(#{self.id}  "
            f"A={self.area}  P={self.perimeter}  c={self.compactness:.3f})"
        )

## GENERATION ALEATOIRE DE BLOC GEOMETRIQUES

In [ ]:
def _neighbors(cell: Cell) -> set[Cell]:
    """Retourne les 4 voisins orthogonaux d'une cellule."""
    x, y = cell
    return {(x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)}


def _generate_one(size: int, rng: random.Random) -> set[Cell]:
    """
    Génère un polyomino aléatoire connexe de `size` cellules
    par marche aléatoire sur la grille.
    """
    cells: set[Cell] = {(0, 0)}
    frontier: list[Cell] = list(_neighbors((0, 0)))

    while len(cells) < size and frontier:
        cell = rng.choice(frontier)
        cells.add(cell)
        for nb in _neighbors(cell):
            if nb not in cells and nb not in frontier:
                frontier.append(nb)
        frontier = [c for c in frontier if c not in cells]

    return cells


def generate_shapes(
    n: int,
    min_size: int = 3,
    max_size: int = 8,
    seed: Optional[int] = None,
) -> list[Shape]:
    """
    Génère `n` polyominos aléatoires, espacés horizontalement pour la visualisation.

    Paramètres
    ----------
    n        : nombre de formes à générer
    min_size : taille minimale d'une pièce (en cellules)
    max_size : taille maximale d'une pièce (en cellules)
    seed     : graine pour la reproductibilité (None = aléatoire)

    Retourne
    --------
    liste de Shape, identifiants 0 à n-1
    """
    rng = random.Random(seed)
    palette = [
        "#FF6B6B", "#4ECDC4", "#FFE66D", "#A29BFE", "#FD79A8",
        "#6C5CE7", "#00B894", "#FDCB6E", "#E17055", "#74B9FF",
        "#55EFC4", "#B2BEC3", "#FAB1A0", "#00CEC9", "#D63031",
    ]

    shapes: list[Shape] = []
    offset_x = 0

    for i in range(n):
        size = rng.randint(min_size, max_size)
        raw = _generate_one(size, rng)
        min_x = min(x for x, _ in raw)
        min_y = min(y for _, y in raw)
        cells = frozenset((x - min_x + offset_x, y - min_y) for x, y in raw)
        offset_x = max(x for x, _ in cells) + 3
        shapes.append(Shape(i, cells, palette[i % len(palette)]))

    return shapes

## TOPOLOGIE DES SOMMETS : POINTES & CREUX 

In [ ]:

@dataclass
class Vertex:
    """
    Sommet d'intérêt géométrique sur le contour d'un polyomino.

    Attributs
    ---------
    cx, cy   : coordonnées du coin (sur la grille des points)
    vtype    : 'pointe' (convexe, 90°) ou 'creux' (concave, 270°)
    quadrant : quadrant Q0–Q3 dans lequel se trouve la cellule associée
    order    : ω = (d1, d2) longueurs des segments de frontière adjacents

    Notes
    -----
    Convention des quadrants pour un coin (cx, cy) :
      Q0 = cellule (cx-1, cy-1)    Q1 = cellule (cx,   cy-1)
      Q2 = cellule (cx-1, cy  )    Q3 = cellule (cx,   cy  )
    """

    cx: int
    cy: int
    vtype: str       # 'pointe' | 'creux'
    quadrant: int    # 0, 1, 2 ou 3
    order: tuple[int, int]

    def order_product(self) -> int:
        """d1 × d2 : aire d'influence locale du sommet."""
        return self.order[0] * self.order[1]


def _boundary_run(
    cells: frozenset[Cell],
    vx: int,
    vy: int,
    dx: int,
    dy: int,
) -> int:
    """
    Longueur du segment rectiligne de frontière partant du coin (vx, vy)
    dans la direction (dx, dy), tant que le segment reste sur la frontière du bloc.

    Directions :
      dx=±1, dy=0  → arête horizontale (longueur mesurée en unités)
      dx=0,  dy=±1 → arête verticale
    """
    length = 0
    x, y = vx, vy

    while True:
        nx, ny = x + dx, y + dy
        if dy == 0:
            # Arête horizontale : séparant (min(x,nx), y-1) et (min(x,nx), y)
            cx_cell = min(x, nx)
            on_boundary = ((cx_cell, y - 1) in cells) != ((cx_cell, y) in cells)
        else:
            # Arête verticale : séparant (x-1, min(y,ny)) et (x, min(y,ny))
            cy_cell = min(y, ny)
            on_boundary = ((x - 1, cy_cell) in cells) != ((x, cy_cell) in cells)

        if not on_boundary:
            break
        length += 1
        x, y = nx, ny

    return length


def get_vertices(shape: Shape) -> list[Vertex]:
    """
    Calcule tous les sommets d'intérêt (pointes et creux) d'un polyomino.

    Algorithme
    ----------
    Pour chaque coin (cx, cy) partagé par des cellules du bloc :
      - On compte les cellules présentes parmi les 4 quadrants adjacents.
      - count = 1 → Pointe (sommet convexe, angle intérieur 90°)
      - count = 3 → Creux  (sommet concave, angle intérieur 270°)
      - count ∈ {0, 2, 4} → coin neutre (ignoré)

    L'ordre ω = (d1, d2) est calculé par `_boundary_run`.
    """
    cells = shape.cells
    has = lambda x, y: (x, y) in cells  # noqa: E731

    # Collecte de tous les coins (angles) de la frontière
    corners: set[Cell] = set()
    for x, y in cells:
        for ax in range(2):
            for ay in range(2):
                corners.add((x + ax, y + ay))

    vertices: list[Vertex] = []
    for cx, cy in corners:
        q = [
            1 if has(cx - 1, cy - 1) else 0,  # Q0
            1 if has(cx,     cy - 1) else 0,  # Q1
            1 if has(cx - 1, cy    ) else 0,  # Q2
            1 if has(cx,     cy    ) else 0,  # Q3
        ]
        n_occ = sum(q)

        if n_occ == 1:
            vtype, quadrant = "pointe", q.index(1)
        elif n_occ == 3:
            vtype, quadrant = "creux", q.index(0)
        else:
            continue  # coin neutre : ignoré

        # Calcul de l'ordre ω = (d_horizontal, d_vertical)
        d_h = (
            _boundary_run(cells, cx, cy, +1, 0)
            + _boundary_run(cells, cx, cy, -1, 0)
        )
        d_v = (
            _boundary_run(cells, cx, cy, 0, +1)
            + _boundary_run(cells, cx, cy, 0, -1)
        )
        vertices.append(
            Vertex(cx, cy, vtype, quadrant, (max(d_h, 1), max(d_v, 1)))
        )

    return vertices

## FUSION ELEMENTAIRE PAR ALIGNEMENT 

In [ ]:

@dataclass
class FusionCandidate:
    """
    Résultat d'une tentative de fusion entre deux blocs.

    Attributs
    ---------
    shape       : bloc résultant de la fusion
    compactness : c(f) du bloc résultant
    pointe      : sommet pointe utilisé pour l'alignement (None si fusion par surface)
    creux       : sommet creux utilisé pour l'alignement (None si fusion par surface)
    rotation    : rotation appliquée à f2 (0–3, en multiples de 90°)
    """

    shape: Shape
    compactness: float
    pointe: Optional[Vertex]
    creux: Optional[Vertex]
    rotation: int


def _try_align(
    f1: Shape,
    f2_rot: Shape,
    p: Vertex,
    c: Vertex,
    rotation: int,
) -> Optional[FusionCandidate]:
    """
    Tente de fusionner f2_rot dans f1 en translatant f2_rot pour que le coin `c`
    coïncide avec le coin `p`. Retourne None si un chevauchement est détecté.
    """
    dx, dy = p.cx - c.cx, p.cy - c.cy
    f2_moved = f2_rot.translated(dx, dy)

    if f1.overlaps(f2_moved):
        return None

    merged = f1.merged_with(f2_moved)
    return FusionCandidate(merged, merged.compactness, p, c, rotation)


def all_valid_pairs(f1: Shape, f2: Shape) -> list[FusionCandidate]:
    """
    Énumère tous les couples (pointe, creux) admissibles entre f1 et f2
    sur les 4 rotations de f2, sans chevauchement.

    Cas 1 : pointe de f1 ↔ creux  de f2_rot (même quadrant)
    Cas 2 : creux  de f1 ↔ pointe de f2_rot (même quadrant)
    """
    candidates: list[FusionCandidate] = []
    v1 = get_vertices(f1)
    pointes1 = [v for v in v1 if v.vtype == "pointe"]
    creux1   = [v for v in v1 if v.vtype == "creux"]

    for r in range(4):
        f2r = f2.rotated(r)
        v2 = get_vertices(f2r)
        pointes2 = [v for v in v2 if v.vtype == "pointe"]
        creux2   = [v for v in v2 if v.vtype == "creux"]

        for p in pointes1:
            for c in creux2:
                if p.quadrant == c.quadrant:
                    result = _try_align(f1, f2r, p, c, r)
                    if result:
                        candidates.append(result)

        for c in creux1:
            for p in pointes2:
                if p.quadrant == c.quadrant:
                    result = _try_align(f1, f2r, c, p, r)
                    if result:
                        candidates.append(result)

    return candidates

## FUSION PAR LONGUEUR DE SURFACE PROCHE 

In [ ]:
def _contact_edges(f1: Shape, f2_moved: Shape) -> int:
    """Compte les arêtes de contact entre f1 et f2_moved (adjacence sans chevauchement)."""
    count = 0
    for x, y in f1.cells:
        for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            if (x + dx, y + dy) in f2_moved.cells:
                count += 1
    return count


def _h_runs(shape: Shape) -> list[tuple[int, int, int]]:
    """Retourne les runs de bords horizontaux : (y_coin, x_start, longueur)."""
    cells, runs = shape.cells, []
    for y in sorted({y for _, y in cells}):
        for dy in [0, 1]:
            y_corner = y + dy
            row = sorted(x for x, yy in cells if yy == y)
            if not row:
                continue
            start, length = row[0], 1
            for i in range(1, len(row)):
                if row[i] == row[i - 1] + 1:
                    length += 1
                else:
                    runs.append((y_corner, start, length))
                    start, length = row[i], 1
            runs.append((y_corner, start, length))
    return runs


def _v_runs(shape: Shape) -> list[tuple[int, int, int]]:
    """Retourne les runs de bords verticaux : (x_coin, y_start, longueur)."""
    cells, runs = shape.cells, []
    for x in sorted({x for x, _ in cells}):
        for dx in [0, 1]:
            x_corner = x + dx
            col = sorted(y for xx, y in cells if xx == x)
            if not col:
                continue
            start, length = col[0], 1
            for i in range(1, len(col)):
                if col[i] == col[i - 1] + 1:
                    length += 1
                else:
                    runs.append((x_corner, start, length))
                    start, length = col[i], 1
            runs.append((x_corner, start, length))
    return runs


def merge_by_surface(f1: Shape, f2: Shape) -> Optional[FusionCandidate]:
    """
    Utilisé quand aucun creux n'est présent dans f1 ni f2.
    Aligne les runs de frontière de longueurs les plus proches et maximise
    le nombre d'arêtes de contact.
    """
    best_contact = -1
    best: Optional[FusionCandidate] = None
    h_runs1, v_runs1 = _h_runs(f1), _v_runs(f1)

    for r in range(4):
        f2r = f2.rotated(r)

        for yc1, xs1, l1 in h_runs1:
            for yc2, xs2, l2 in _h_runs(f2r):
                if abs(l1 - l2) > max(l1, l2):
                    continue
                dx = xs1 - xs2
                for adj_dy in [yc1 - yc2 - 1, yc1 - yc2]:
                    f2m = f2r.translated(dx, adj_dy)
                    if f1.overlaps(f2m):
                        continue
                    c = _contact_edges(f1, f2m)
                    if c > best_contact:
                        best_contact = c
                        mg = f1.merged_with(f2m)
                        best = FusionCandidate(mg, mg.compactness, None, None, r)

        for xc1, ys1, l1 in v_runs1:
            for xc2, ys2, l2 in _v_runs(f2r):
                if abs(l1 - l2) > max(l1, l2):
                    continue
                dy = ys1 - ys2
                for adj_dx in [xc1 - xc2 - 1, xc1 - xc2]:
                    f2m = f2r.translated(adj_dx, dy)
                    if f1.overlaps(f2m):
                        continue
                    c = _contact_edges(f1, f2m)
                    if c > best_contact:
                        best_contact = c
                        mg = f1.merged_with(f2m)
                        best = FusionCandidate(mg, mg.compactness, None, None, r)

    return best


def _best(candidates: list[FusionCandidate]) -> Optional[FusionCandidate]:
    """Retourne le candidat de compacité maximale, ou None si la liste est vide."""
    return max(candidates, key=lambda c: c.compactness) if candidates else None

## HEURISTIQUES DE SELECTIONS DE FUSION

### HEURISTIQUE 1 - Ordre maximal

In [ ]:
def heuristic_1_candidates(f1: Shape, f2: Shape) -> list[FusionCandidate]:
    """
    H1 — Sélection par ordre maximal.

    Pour chaque rotation de f2 :
      Cas A : pointe de f1 d'ordre max (d1×d2) ↔ creux  de f2 de même quadrant d'ordre max
      Cas B : creux  de f1 d'ordre max          ↔ pointe de f2 de même quadrant d'ordre max

    Ne teste qu'un seul couple par cas/rotation → complexité O(V).

    Correction clé : le filtrage par quadrant garantit la compatibilité géométrique.
    Sans ce filtrage, les sommets d'ordres maximaux peuvent avoir des quadrants opposés,
    entraînant un chevauchement systématique.
    """

    def argmax_order(verts: list[Vertex]) -> Optional[Vertex]:
        return max(verts, key=lambda v: v.order_product(), default=None)

    def argmax_compatible(verts: list[Vertex], quadrant: int) -> Optional[Vertex]:
        """Parmi les sommets de quadrant `quadrant`, retourne celui d'ordre max."""
        compatible = [v for v in verts if v.quadrant == quadrant]
        return argmax_order(compatible)

    candidates: list[FusionCandidate] = []
    v1 = get_vertices(f1)
    pointes1 = [v for v in v1 if v.vtype == "pointe"]
    creux1   = [v for v in v1 if v.vtype == "creux"]

    for r in range(4):
        f2r = f2.rotated(r)
        v2 = get_vertices(f2r)
        pointes2 = [v for v in v2 if v.vtype == "pointe"]
        creux2   = [v for v in v2 if v.vtype == "creux"]

        # Cas A : pointe max de f1 → creux compatible max de f2
        p_max = argmax_order(pointes1)
        if p_max:
            c_compat = argmax_compatible(creux2, p_max.quadrant)
            if c_compat:
                result = _try_align(f1, f2r, p_max, c_compat, r)
                if result:
                    candidates.append(result)

        # Cas B : creux max de f1 → pointe compatible max de f2
        c_max = argmax_order(creux1)
        if c_max:
            p_compat = argmax_compatible(pointes2, c_max.quadrant)
            if p_compat:
                result = _try_align(f1, f2r, c_max, p_compat, r)
                if result:
                    candidates.append(result)

    return candidates

### HEURISTIQUE 2 - Distance-produit d'ordres

In [ ]:
def heuristic_2_candidates(f1: Shape, f2: Shape) -> list[FusionCandidate]:
    """
    H2 — Sélection par minimisation de l'écart dimensionnel et maximisation
    de la surface de contact.

    Étape 1 — Construction de l'ensemble I :
      I = { i | Δω[0] = min_j Δω[0]  OU  Δω[1] = min_j Δω[1] }
      (couples dont les ordres sont les plus proches dimensionnellement)

    Étape 2 — Affinage vers I' :
      I' = { i ∈ I | prod[0] = max prod[0]  OU  prod[1] = max prod[1] }
      (parmi I, les couples de plus grande aire d'influence)

    Complexité : O(V²) — explore tous les couples valides avant filtrage.

    Repli automatique sur `merge_by_surface` si aucun creux n'existe.
    """
    candidates = all_valid_pairs(f1, f2)
    if not candidates:
        result = merge_by_surface(f1, f2)
        return [result] if result else []

    def delta(fc: FusionCandidate) -> tuple[int, int]:
        if fc.pointe is None or fc.creux is None:
            return (0, 0)
        return (
            abs(fc.pointe.order[0] - fc.creux.order[0]),
            abs(fc.pointe.order[1] - fc.creux.order[1]),
        )

    def product(fc: FusionCandidate) -> tuple[int, int]:
        if fc.pointe is None or fc.creux is None:
            return (0, 0)
        return (
            fc.pointe.order[0] * fc.creux.order[0],
            fc.pointe.order[1] * fc.creux.order[1],
        )

    # Construction de I
    deltas = [delta(c) for c in candidates]
    min_d0 = min(d[0] for d in deltas)
    min_d1 = min(d[1] for d in deltas)
    I = (
        [c for c, d in zip(candidates, deltas) if d[0] == min_d0 or d[1] == min_d1]
        or candidates
    )

    # Affinage vers I'
    products = [product(c) for c in I]
    max_p0   = max(p[0] for p in products)
    max_p1   = max(p[1] for p in products)
    I_prime  = (
        [c for c, p in zip(I, products) if p[0] == max_p0 or p[1] == max_p1]
        or I
    )

    return I_prime

### HEURISTIQUE 3 - Brute Force gloutonne

In [ ]:
def heuristic_3_candidates(f1: Shape, f2: Shape) -> list[FusionCandidate]:
    """
    H3 — Force brute gloutonne : évalue tous les couples (pointe, creux)
    valides sur les 4 rotations de f2 sans aucun filtrage.

    Sert de référentiel absolu pour évaluer H1 et H2.
    Complexité : O(V1 × V2 × 4).

    Repli automatique sur `merge_by_surface` si aucun creux n'existe.
    """
    v = get_vertices(f1) + get_vertices(f2)

    if not any(vertex.vtype == "creux" for vertex in v):
        result = merge_by_surface(f1, f2)
        return [result] if result else []

    candidates = all_valid_pairs(f1, f2)
    if candidates:
        return candidates

    # Repli : pas de couple pointe-creux valide → fusion par surface
    result = merge_by_surface(f1, f2)
    return [result] if result else []

## ALGORITHME GLOUTON MULTI-BRANCHES

In [ ]:

@dataclass
class StepLog:
    """
    Enregistrement d'une étape de l'algorithme glouton.

    Attributs
    ---------
    step      : numéro d'étape (1-indexé)
    step_type : 'fusion' | 'local_optimum' | 'done'
    f1_id     : identifiant du bloc le plus compact sélectionné
    f2_id     : identifiant du second bloc sélectionné
    c_before  : compacités de f1 et f2 avant fusion
    c_after   : compacité du bloc résultant (0.0 si optimum local)
    message   : description textuelle de l'étape
    """

    step: int
    step_type: str
    f1_id: int
    f2_id: int
    c_before: tuple[float, float]
    c_after: float
    message: str


def _state_key(shapes: list[Shape]) -> frozenset:
    """Clé d'état unique : ensemble des cellules de chaque forme (indépendant de l'ordre)."""
    return frozenset(s.cells for s in shapes)


# ── Paramètre global du beam search ──────────────────────────────────────
#   0 → illimité (recherche exacte, attention à la mémoire pour n > 8)
#   N → conserver les N meilleures branches actives à chaque étape
MAX_BRANCHES: int = 150

# ── Tables de dispatch ────────────────────────────────────────────────────
_HEURISTIC_FN: dict = {
    1: heuristic_1_candidates,
    2: heuristic_2_candidates,
    3: heuristic_3_candidates,
}

_HEURISTIC_LABEL: dict[int, str] = {
    1: "H1 — Ordre maximal",
    2: "H2 — Distance-produit",
    3: "H3 — Force brute gloutonne",
}


def greedy_compact_assembly(
    shapes: list[Shape],
    heuristic: int = 3,
    verbose: bool = True,
) -> tuple[list[Shape], list[StepLog], list[list[StepLog]], list[Shape]]:
    """
    Algorithme glouton multi-branches avec beam search.

    À chaque étape :
      1. Les 2 formes de compacité maximale sont sélectionnées (f1, f2).
      2. L'heuristique `heuristic` génère N candidats de fusion.
      3. Chaque candidat ouvre une branche indépendante.
      4. Le beam search élague les branches en excès de MAX_BRANCHES.

    Les états identiques (même ensemble de cellules) sont dédupliqués.

    Paramètres
    ----------
    shapes    : liste des blocs à assembler
    heuristic : 1 → H1, 2 → H2, 3 → H3
    verbose   : afficher la progression dans la console

    Retourne
    --------
    best_shapes  : formes finales de la meilleure branche (compacité max)
    best_logs    : journal de cette branche
    all_branches : journaux de toutes les branches (pour la visualisation)
    best_trace   : liste des formes fusionnées intermédiaires (meilleure branche)
    """
    sep = "=" * 65
    label = _HEURISTIC_LABEL.get(heuristic, str(heuristic))
    fn    = _HEURISTIC_FN.get(heuristic)
    if fn is None:
        raise ValueError(f"Heuristique inconnue : {heuristic}. Valeurs valides : 1, 2, 3.")

    if verbose:
        print(sep)
        print(f"  GLOUTON MULTI-BRANCHES — {label}")
        print(
            f"  {len(shapes)} formes  |  MAX_BRANCHES="
            f"{'∞' if MAX_BRANCHES == 0 else MAX_BRANCHES}"
        )
        print(sep)

    # État : StateKey → (formes_courantes, logs, trace_formes_fusionnées)
    active: dict = {_state_key(shapes): (list(shapes), [], [])}
    step = 0

    def _is_stuck(logs: list[StepLog]) -> bool:
        """True si la dernière entrée du journal est un optimum local."""
        return bool(logs) and logs[-1].step_type == "local_optimum"

    def _score(item: tuple) -> float:
        """Score d'une branche = compacité maximale de ses formes courantes."""
        return max(s.compactness for s in item[0])

    def _beam_trim(d: dict) -> dict:
        """
        Élague le dictionnaire de branches actives pour ne garder que MAX_BRANCHES.
        Les branches terminées (len == 1) ou bloquées sont toujours conservées.
        """
        if MAX_BRANCHES == 0 or len(d) <= MAX_BRANCHES:
            return d
        done = {k: v for k, v in d.items() if len(v[0]) <= 1 or _is_stuck(v[1])}
        actv = {k: v for k, v in d.items() if k not in done}
        sorted_active = sorted(actv.items(), key=lambda kv: _score(kv[1]), reverse=True)
        quota = max(1, MAX_BRANCHES - len(done))
        return dict(list(done.items()) + sorted_active[:quota])

    def _get_candidates(fi: Shape, fj: Shape) -> list[FusionCandidate]:
        """Appelle l'heuristique, avec repli sur merge_by_surface si aucun creux."""
        v = get_vertices(fi) + get_vertices(fj)
        if not any(x.vtype == "creux" for x in v):
            result = merge_by_surface(fi, fj)
            return [result] if result else []
        return fn(fi, fj)

    # ── Boucle principale ─────────────────────────────────────────────────
    while True:
        to_merge = {
            k: v for k, v in active.items()
            if len(v[0]) > 1 and not _is_stuck(v[1])
        }
        if not to_merge:
            break

        step += 1
        new_active = {
            k: v for k, v in active.items()
            if len(v[0]) <= 1 or _is_stuck(v[1])
        }

        for key, (current, logs, trace) in to_merge.items():
            # Sélection des deux blocs les plus compacts
            sorted_shapes = sorted(current, key=lambda x: x.compactness, reverse=True)
            f1, f2, rest = sorted_shapes[0], sorted_shapes[1], sorted_shapes[2:]
            c1, c2 = f1.compactness, f2.compactness
            candidates = _get_candidates(f1, f2)

            if not candidates:
                log = StepLog(
                    step, "local_optimum", f1.id, f2.id, (c1, c2), 0.0,
                    f"  [{step:>3}] Optimum local — {len(current)} forme(s) non fusionnée(s)",
                )
                if key not in new_active:
                    new_active[key] = (current, logs + [log], trace)
                continue

            for cand in candidates:
                fused = Shape(f1.id, cand.shape.cells, f1.color)
                new_shapes = [fused] + list(rest)
                new_key    = _state_key(new_shapes)
                log = StepLog(
                    step, "fusion", f1.id, f2.id, (c1, c2),
                    fused.compactness,
                    f"  [{step:>3}] #{f1.id} + #{f2.id}"
                    f" → c={fused.compactness:.4f}  [{len(new_shapes)} restant(s)]",
                )
                new_logs  = logs + [log]
                new_trace = trace + [fused]

                if new_key not in new_active:
                    new_active[new_key] = (new_shapes, new_logs, new_trace)
                else:
                    # Déduplications : garder la branche la plus compacte
                    existing_c = max(x.compactness for x in new_active[new_key][0])
                    if fused.compactness > existing_c:
                        new_active[new_key] = (new_shapes, new_logs, new_trace)

        active = _beam_trim(new_active)

        if verbose:
            n_done = sum(1 for v in active.values() if len(v[0]) == 1)
            print(f"  Étape {step} : {len(active)} branche(s)  ({n_done} terminée(s))")

    # ── Finalisation ──────────────────────────────────────────────────────
    final: dict = {}
    for key, (sh, lg, tr) in active.items():
        if len(sh) == 1:
            s = sh[0]
            lg = lg + [StepLog(
                step + 1, "done", -1, -1, (0.0, 0.0), s.compactness,
                f"       ✔  c={s.compactness:.4f}  A={s.area}  P={s.perimeter}",
            )]
        final[key] = (sh, lg, tr)

    def _score_item(item: tuple) -> float:
        return max(s.compactness for s in item[0])

    best_shapes, best_logs, best_trace = max(final.values(), key=_score_item)
    c_best = max(s.compactness for s in best_shapes)

    if verbose:
        print(sep)
        print(f"  {len(final)} branche(s)  |  meilleure c = {c_best:.4f}")
        if len(best_shapes) == 1:
            s = best_shapes[0]
            print(f"  ✔  Assemblage complet  A={s.area}  P={s.perimeter}")
        else:
            print(f"  ⚠  {len(best_shapes)} forme(s) non fusionnée(s)")
        print(sep)

    all_branches = [v[1] for v in final.values()]
    return best_shapes, best_logs, all_branches, best_trace

## METRIQUE DE RESEMBLANCE MORPHOLOGIQUE

In [ ]:

def _align_shape(reference: Shape, candidate: Shape) -> frozenset[Cell]:
    """
    Recherche la rotation r ∈ {0°, 90°, 180°, 270°} et la translation (dx, dy)
    qui minimisent le cardinal de la différence symétrique entre `reference` et
    `candidate` aligné.

    Stratégie : pour chaque rotation, on teste toutes les translations qui
    superposent au moins une cellule de `candidate` sur une cellule de `reference`.

    Retourne les cellules de `candidate` après transformation optimale.
    """
    ref_cells  = reference.cells
    best_diff  = len(ref_cells) + len(candidate.cells) + 1  # pire cas + 1
    best_cells = candidate.cells

    for r in range(4):
        rot_cells = candidate.rotated(r).normalized().cells
        for rx, ry in ref_cells:
            for cx, cy in rot_cells:
                dx, dy = rx - cx, ry - cy
                moved = frozenset((x + dx, y + dy) for x, y in rot_cells)
                sym_diff = len(ref_cells.symmetric_difference(moved))
                if sym_diff < best_diff:
                    best_diff  = sym_diff
                    best_cells = moved

    return best_cells


def resemblance_simple(shape_a: Shape, shape_b: Shape) -> float:
    """
    Métrique de ressemblance simple (non pondérée).

    R = N_diff / A_tot = |A Δ B| / |A ∪ B|

    où A Δ B est la différence symétrique après alignement optimal.
    R ∈ [0, 1] : 0 → formes identiques, 1 → aucune cellule commune.
    """
    aligned_b = _align_shape(shape_a, Shape(shape_b.id, shape_b.cells))
    union      = shape_a.cells | aligned_b
    sym_diff   = shape_a.cells.symmetric_difference(aligned_b)
    a_tot      = len(union)
    return len(sym_diff) / a_tot if a_tot > 0 else 0.0


def _cell_dist_to_shape(cell: Cell, shape_cells: frozenset[Cell]) -> int:
    """
    Distance de Manhattan minimale entre `cell` et l'ensemble `shape_cells`.
    Utilisée comme degré d'excentricité w_i dans la métrique pondérée.
    """
    cx, cy = cell
    return min(abs(cx - sx) + abs(cy - sy) for sx, sy in shape_cells)


def resemblance_weighted(shape_a: Shape, shape_b: Shape) -> float:
    """
    Métrique de ressemblance pondérée avec degré d'excentricité.

    R_w = Σ w_i / A_tot

    où w_i = d_Manhattan(cellule_i, périmètre de shape_a) + 1
      (+1 garantit w_i ≥ 1 même pour les cellules adjacentes au périmètre)

    Les cellules divergentes éloignées du périmètre de référence sont
    davantage pénalisées que les cellules proches, traduisant une divergence
    morphologique plus sévère.
    """
    aligned_b = _align_shape(shape_a, Shape(shape_b.id, shape_b.cells))
    union      = shape_a.cells | aligned_b
    sym_diff   = shape_a.cells.symmetric_difference(aligned_b)
    a_tot      = len(union)

    if a_tot == 0:
        return 0.0

    weighted_sum = sum(
        _cell_dist_to_shape(cell, shape_a.cells) + 1
        for cell in sym_diff
    )
    return weighted_sum / a_tot


def compute_resemblance(
    results: dict[int, tuple],
) -> dict[int, dict[str, float]]:
    """
    Calcule R_simple et R_pondéré pour H1 et H2 par rapport à H3 (référence fixe).

    H3 est exclu du résultat (référence = 0 par définition).

    Paramètres
    ----------
    results : dict { heuristic_id: (best_shapes, logs, branches, trace) }

    Retourne
    --------
    dict { heuristic_id: {"simple": R_s, "weighted": R_w} }
    """
    ref_shapes = results[3][0]
    ref_shape  = Shape(-1, frozenset(c for s in ref_shapes for c in s.cells))

    metrics: dict[int, dict[str, float]] = {}
    for hi, (best, *_) in results.items():
        if hi == 3:
            continue
        cand_shape = Shape(-1, frozenset(c for s in best for c in s.cells))
        metrics[hi] = {
            "simple":   resemblance_simple(ref_shape, cand_shape),
            "weighted": resemblance_weighted(ref_shape, cand_shape),
        }
    return metrics

## VISUALISATION

### palette de couleurs

In [ ]:
BG     = "white"
PANEL  = "#f8f9fa"
BORDER = "#ced4da"
TEXT   = "#212529"
MUTED  = "#6c757d"

C_H1 = "#e67700"  # orange   — H1
C_H2 = "#1971c2"  # bleu     — H2
C_H3 = "#7048e8"  # violet   — H3

_COLOR  = {1: C_H1, 2: C_H2, 3: C_H3}
_LS     = {1: "--",  2: "-",   3: "-."}
_MARKER = {1: "o",   2: "o",   3: "D"}
_LABEL  = {
    1: "H1 — Ordre maximal",
    2: "H2 — Distance-produit",
    3: "H3 — Force brute gloutonne",
}

### HELPERS DE STYLE

In [ ]:

def _style_ax(ax: plt.Axes) -> None:
    """Applique le style commun (fond, grille, bordures)."""
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)
        sp.set_linewidth(0.8)
    ax.tick_params(colors=TEXT, labelsize=8)
    ax.grid(True, color=BORDER, linewidth=0.5, alpha=0.9, zorder=0)


def _plot_branches(
    ax: plt.Axes,
    all_branches: list[list[StepLog]],
    c_init: float,
    color: str,
    alpha: float = 0.13,
    lw: float = 0.6,
) -> None:
    """Trace toutes les branches en filigrane (lignes fines et semi-transparentes)."""
    for logs in all_branches:
        fusions = [l for l in logs if l.step_type == "fusion"]
        if not fusions:
            continue
        xs = list(range(len(fusions) + 1))
        ys = [c_init] + [l.c_after for l in fusions]
        ax.plot(xs, ys, color=color, linewidth=lw, alpha=alpha, linestyle="-", zorder=2)


def _plot_best(
    ax: plt.Axes,
    logs: list[StepLog],
    c_init: float,
    color: str,
    label: str,
    linestyle: str = "-",
    marker: str = "o",
    lw: float = 2.3,
    ms: float = 9,
) -> None:
    """Trace la meilleure branche en gras, avec annotation de la valeur finale."""
    fusions = [l for l in logs if l.step_type == "fusion"]
    if not fusions:
        return
    xs = list(range(len(fusions) + 1))
    ys = [c_init] + [l.c_after for l in fusions]
    ax.plot(
        xs, ys,
        linestyle=linestyle, marker=marker, color=color,
        linewidth=lw, markersize=ms,
        markerfacecolor="white", markeredgecolor=color, markeredgewidth=2.0,
        label=label, zorder=6,
    )
    ax.annotate(
        f"  {ys[-1]:.4f}", xy=(xs[-1], ys[-1]),
        fontsize=8.5, color=color, fontfamily="monospace", va="center", zorder=7,
    )


def _finalise_evo_ax(ax: plt.Axes, c_init: float, title: str) -> None:
    """Ajoute la légende, les labels et le titre à un axe d'évolution."""
    ax.axhline(c_init, color=MUTED, linewidth=1.0, linestyle=":",
               label=f"c̄ initiale = {c_init:.3f}", zorder=1)
    ax.set_xlabel("Étape de fusion", fontsize=10, color=TEXT, labelpad=7)
    ax.set_ylabel("c(f) = Aire / Périmètre", fontsize=10, color=TEXT, labelpad=7)
    ax.legend(fontsize=9, facecolor="white", edgecolor=BORDER,
              labelcolor=TEXT, framealpha=1.0, loc="upper left", frameon=True)
    ax.tick_params(colors=TEXT, labelsize=9)
    ax.set_title(title, color=TEXT, fontsize=11, fontweight="bold", pad=10, loc="left")


def _fig_header(fig: plt.Figure, title: str, subtitle: str) -> None:
    """Ajoute un titre et un sous-titre centrés en haut de la figure."""
    fig.suptitle(title, color=TEXT, fontsize=13, fontweight="bold", y=0.97)
    fig.text(0.5, 0.925, subtitle, ha="center", color=MUTED, fontsize=8.5)


def _draw_cells(
    ax: plt.Axes,
    shapes: list[Shape],
    color: str,
    label: Optional[str] = None,
    lw_border: float = 1.8,
) -> None:
    """Dessine les cellules d'une liste de formes, centrées dans l'axe."""
    all_cells = [(x, y) for s in shapes for x, y in s.cells]
    if not all_cells:
        return

    min_x = min(x for x, _ in all_cells)
    min_y = min(y for _, y in all_cells)
    centered = [(x - min_x, y - min_y) for x, y in all_cells]

    face = to_rgba(color, alpha=0.60)
    edge = to_rgba(color, alpha=1.00)
    for x, y in centered:
        ax.add_patch(mpatches.FancyBboxPatch(
            (x + 0.06, y + 0.06), 0.88, 0.88,
            boxstyle="round,pad=0.04",
            facecolor=face, edgecolor=edge, linewidth=1.2, zorder=3,
        ))

    xs  = [x for x, _ in centered]
    ys  = [y for _, y in centered]
    pad = 1.5
    ax.set_xlim(min(xs) - pad, max(xs) + 1 + pad)
    ax.set_ylim(min(ys) - pad, max(ys) + 1 + pad)
    ax.set_aspect("equal")
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_edgecolor(color)
        sp.set_linewidth(lw_border)
    ax.grid(True, color=BORDER, linewidth=0.4, alpha=0.8, zorder=0)
    ax.tick_params(labelbottom=False, labelleft=False, bottom=False, left=False)

    if label:
        ax.set_title(label, color=color, fontsize=10, fontweight="bold", pad=6)


def _shape_info(
    shapes: list[Shape],
    logs: list[StepLog],
) -> tuple[int, bool, int, int, float]:
    """Retourne (n_fusions, complet, aire, périmètre, compacité) pour un résultat."""
    n_fusions = sum(1 for l in logs if l.step_type == "fusion")
    success   = len(shapes) == 1
    merged    = Shape(-1, frozenset(c for s in shapes for c in s.cells))
    a, p      = merged.area, merged.perimeter
    return n_fusions, success, a, p, (a / p if p else 0.0)

### GRAPHIQUES D'EVOLUTION

In [ ]:
def _save_evolution_global(
    initial: list[Shape],
    results: dict[int, tuple],
    filepath: str,
) -> None:
    """Graphe global : une courbe par heuristique (meilleure branche uniquement)."""
    fig, ax = plt.subplots(figsize=(13, 7), facecolor=BG)
    fig.subplots_adjust(left=0.09, right=0.96, top=0.87, bottom=0.12)
    _style_ax(ax)
    c_init = sum(s.compactness for s in initial) / len(initial)

    for hi in sorted(results):
        _, logs, _, _ = results[hi]
        _plot_best(ax, logs, c_init, _COLOR[hi], _LABEL[hi], _LS[hi], _MARKER[hi])

    _finalise_evo_ax(ax, c_init, "Comparaison H1 / H2 / H3 — meilleure compacité atteinte")
    _fig_header(
        fig,
        "Évolution de la compacité au fil des fusions",
        f"{len(initial)} formes  ·  seed={CONFIG.get('seed', '—')}  ·  "
        f"taille {CONFIG.get('min_size')}–{CONFIG.get('max_size')}",
    )
    plt.savefig(filepath, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
    print(f"  [évolution globale]      → {filepath}")


def _save_comparaison(
    initial: list[Shape],
    hi_a: int,
    hi_b: int,
    results: dict[int, tuple],
    filepath: str,
) -> None:
    """Branches filigranées + meilleure courbe pour deux heuristiques."""
    fig, ax = plt.subplots(figsize=(11, 6), facecolor=BG)
    fig.subplots_adjust(left=0.10, right=0.97, top=0.87, bottom=0.13)
    _style_ax(ax)
    c_init = sum(s.compactness for s in initial) / len(initial)

    for hi in (hi_a, hi_b):
        _, logs, branches, _ = results[hi]
        _plot_branches(ax, branches, c_init, _COLOR[hi])
        _plot_best(
            ax, logs, c_init, _COLOR[hi],
            _LABEL[hi] + " (meilleure branche)",
            _LS[hi], _MARKER[hi], lw=2.5, ms=10,
        )

    ax.plot([], [], color=MUTED, linewidth=0.7, alpha=0.4, label="Branches explorées")
    lbl_a = _LABEL[hi_a].split("—")[0].strip()
    lbl_b = _LABEL[hi_b].split("—")[0].strip()
    _finalise_evo_ax(ax, c_init, f"{lbl_a}  vs  {lbl_b}  —  branches explorées")
    _fig_header(
        fig, f"Comparaison {lbl_a} / {lbl_b}",
        f"{len(initial)} formes  ·  seed={CONFIG.get('seed', '—')}",
    )
    plt.savefig(filepath, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
    print(f"  [comparaison {lbl_a}/{lbl_b}]  → {filepath}")

### GRAPHIQUES DE FORMES

In [ ]:

def _save_all_shapes(
    initial: list[Shape],
    results: dict[int, tuple],
    filepath: str,
) -> None:
    """Affiche les formes finales de toutes les heuristiques côte à côte."""
    n_panels = len(results)
    fig, axes = plt.subplots(1, n_panels, figsize=(4 * n_panels + 2, 7), facecolor=BG)
    if n_panels == 1:
        axes = [axes]
    fig.subplots_adjust(left=0.01, right=0.99, top=0.82, bottom=0.16, wspace=0.28)

    short_labels = {1: "H1\nOrdre maximal", 2: "H2\nDist-produit", 3: "H3\nForce brute"}

    for ax, hi in zip(axes, sorted(results)):
        best, logs, _, _ = results[hi]
        _draw_cells(ax, best, _COLOR[hi], label=short_labels.get(hi, f"H{hi}"))
        n, ok, a, p, c = _shape_info(best, logs)
        status = "✔ complet" if ok else f"⚠ {len(best)} fragment(s)"
        ax.text(
            0.5, -0.10,
            f"A={a}  P={p}  c={c:.4f}\n{status}  ({n} fusions)",
            transform=ax.transAxes, ha="center", va="top",
            fontsize=7.5, color=MUTED, fontfamily="monospace",
        )

    fig.suptitle(
        "Formes finales — comparaison H1 / H2 / H3",
        color=TEXT, fontsize=13, fontweight="bold", y=0.97,
    )
    fig.text(
        0.5, 0.905,
        f"seed={CONFIG.get('seed', '—')}  ·  "
        f"taille {CONFIG.get('min_size')}–{CONFIG.get('max_size')} cellules",
        ha="center", color=MUTED, fontsize=8.5,
    )
    plt.savefig(filepath, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
    print(f"  [formes finales]         → {filepath}")

### GRAPHIQUES DE RESSEMBLANCES

In [ ]:
def _save_resemblance(
    results: dict[int, tuple],
    metrics: dict[int, dict[str, float]],
    filepath: str,
) -> None:
    """Diagramme en barres : R_simple et R_pondéré de H1 et H2 par rapport à H3."""
    heuristics = sorted(metrics.keys())
    labels     = [_LABEL[hi].split("—")[0].strip() for hi in heuristics]
    colors     = [_COLOR[hi] for hi in heuristics]
    r_simple   = [metrics[hi]["simple"]   for hi in heuristics]
    r_weighted = [metrics[hi]["weighted"] for hi in heuristics]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), facecolor=BG)
    fig.subplots_adjust(left=0.07, right=0.97, top=0.83, bottom=0.14, wspace=0.32)

    panels = [
        (r_simple,   "① Ressemblance simple",              r"$R = N_{diff}\,/\,A_{tot}$"),
        (r_weighted, "② Ressemblance pondérée (excentricité)", r"$R = \sum w_i\,/\,A_{tot}$"),
    ]

    for ax, (values, title, formula) in zip(axes, panels):
        ax.set_facecolor(PANEL)
        for sp in ax.spines.values():
            sp.set_edgecolor(BORDER)
            sp.set_linewidth(0.8)
        ax.tick_params(colors=TEXT, labelsize=9)
        ax.grid(True, color=BORDER, linewidth=0.5, alpha=0.9, axis="y", zorder=0)

        bars = ax.bar(labels, values, color=colors, alpha=0.78,
                      edgecolor=colors, linewidth=1.2, zorder=3, width=0.45)

        for bar, v in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(values, default=0) * 0.02 + 1e-6,
                f"{v:.4f}", ha="center", va="bottom",
                fontsize=9.5, color=TEXT, fontfamily="monospace", fontweight="bold",
            )

        ax.set_ylim(0, max(values, default=0.1) * 1.22 + 1e-6)
        ax.set_ylabel("R  (0 = identique à H3)", fontsize=9.5, color=TEXT, labelpad=6)
        ax.set_title(f"{title}\n{formula}", color=TEXT, fontsize=10.5,
                     fontweight="bold", pad=8)
        ax.tick_params(axis="x", colors=TEXT, labelsize=10)

    fig.suptitle(
        "Métrique de ressemblance  —  H1 et H2 comparés à H3 (référence)",
        color=TEXT, fontsize=12, fontweight="bold", y=0.97,
    )
    fig.text(
        0.5, 0.905,
        "R faible → forme proche de H3  ·  "
        "w_i = distance de Manhattan de la cellule divergente au périmètre de H3",
        ha="center", color=MUTED, fontsize=8.0,
    )
    plt.savefig(filepath, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
    print(f"  [ressemblance finale]    → {filepath}")


def _save_resemblance_evolution(
    results: dict[int, tuple],
    filepath: str,
) -> None:
    """
    Évolution des métriques de ressemblance (simple et pondérée) à chaque étape
    de fusion, pour H1 et H2 par rapport à la forme finale de H3.

    Chaque point correspond à la forme intermédiaire accumulée après k fusions
    sur la meilleure branche de l'heuristique concernée.
    Un R décroissant indique une convergence morphologique vers H3.
    """
    ref_shapes = results[3][0]
    ref_shape  = Shape(-1, frozenset(c for s in ref_shapes for c in s.cells))

    # Calcul des séries temporelles
    series: dict[int, dict[str, list[float]]] = {}
    for hi in (1, 2):
        _, _, _, trace = results[hi]
        series[hi] = {
            "simple":   [resemblance_simple(ref_shape, inter)   for inter in trace],
            "weighted": [resemblance_weighted(ref_shape, inter) for inter in trace],
        }

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=BG)
    fig.subplots_adjust(left=0.08, right=0.97, top=0.85, bottom=0.12, wspace=0.30)

    panels = [
        ("simple",   "① Ressemblance simple   R = N_diff / A_tot"),
        ("weighted", "② Ressemblance pondérée  R = Σ w_i / A_tot"),
    ]

    for ax, (metric, title) in zip(axes, panels):
        ax.set_facecolor(PANEL)
        for sp in ax.spines.values():
            sp.set_edgecolor(BORDER)
            sp.set_linewidth(0.8)
        ax.tick_params(colors=TEXT, labelsize=9)
        ax.grid(True, color=BORDER, linewidth=0.5, alpha=0.9, zorder=0)
        ax.set_xlabel("Étape de fusion", fontsize=10, color=TEXT, labelpad=7)
        ax.set_ylabel("R  (0 = identique à H3)", fontsize=10, color=TEXT, labelpad=7)
        ax.set_title(title, color=TEXT, fontsize=10.5, fontweight="bold", pad=8, loc="left")

        for hi in (1, 2):
            vals = series[hi][metric]
            if not vals:
                continue
            xs = list(range(1, len(vals) + 1))
            ax.plot(
                xs, vals,
                color=_COLOR[hi], linewidth=2.2,
                linestyle=_LS[hi], marker=_MARKER[hi],
                markersize=7, markerfacecolor="white",
                markeredgecolor=_COLOR[hi], markeredgewidth=2.0,
                label=_LABEL[hi], zorder=5,
            )
            ax.annotate(
                f"  {vals[-1]:.4f}", xy=(xs[-1], vals[-1]),
                fontsize=8.5, color=_COLOR[hi], fontfamily="monospace", va="center",
            )

        ax.legend(fontsize=9, facecolor="white", edgecolor=BORDER,
                  labelcolor=TEXT, framealpha=1.0, loc="upper right")

    fig.suptitle(
        "Évolution de la ressemblance morphologique  —  H1 et H2 vs H3",
        color=TEXT, fontsize=12, fontweight="bold", y=0.97,
    )
    fig.text(
        0.5, 0.91,
        "Chaque point = forme accumulée intermédiaire de la meilleure branche"
        "  ·  R décroissant → convergence vers H3",
        ha="center", color=MUTED, fontsize=8.0,
    )
    plt.savefig(filepath, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.show()
    print(f"  [évolution ressemblance] → {filepath}")


def visualize(
    initial: list[Shape],
    results: dict[int, tuple],
    save_path: str = "compacite",
) -> None:
    """
    Génère 4 images à partir des résultats des 3 heuristiques :
      {base}_evolution.png        — courbes d'évolution H1 / H2 / H3
      {base}_comparaison_h2.png   — H2 vs H3 avec branches explorées
      {base}_formes.png           — formes finales côte à côte
    """
    base = save_path.replace(".png", "")
    print()
    _save_evolution_global(initial, results, base + "_evolution.png")
    _save_comparaison(initial, 2, 3, results, base + "_comparaison_h2.png")
    _save_all_shapes(initial, results, base + "_formes.png")

## CONFIGURATIONS ET POINT D'ENTREE

In [ ]:
CONFIG: dict = {
    # ── Instance principale ─────────────────────────────────────────────
    "n_shapes" : 10,   # nombre de polyominos à générer
    "min_size" : 3,     # taille minimale d'une pièce (cellules)
    "max_size" : 8,     # taille maximale d'une pièce (cellules)
    "seed"     : 42,    # graine aléatoire (None = non reproductible)
    "save"     : "compacite",  # préfixe des fichiers images générés
}


def main(
    n_shapes: int           = CONFIG["n_shapes"],
    min_size: int           = CONFIG["min_size"],
    max_size: int           = CONFIG["max_size"],
    seed:     Optional[int] = CONFIG["seed"],
    save:     str           = CONFIG["save"],
) -> None:
    """
    Point d'entrée principal. Compatible script terminal, Jupyter, Kaggle et Colab.

    Exécute les 3 heuristiques sur une instance aléatoire et produit 5 images :
      {save}_evolution.png              — courbes d'évolution H1 / H2 / H3
      {save}_comparaison_h2.png         — H2 vs H3 avec branches
      {save}_formes.png                 — formes finales côte à côte
      {save}_ressemblance.png           — métriques finales de ressemblance
      {save}_ressemblance_evolution.png — évolution temporelle des métriques

    Paramètres
    ----------
    n_shapes : nombre de polyominos à générer
    min_size : taille minimale d'une pièce (en cellules)
    max_size : taille maximale d'une pièce (en cellules)
    seed     : graine aléatoire pour la reproductibilité
    save     : préfixe du nom des fichiers images

    Exemples
    --------
    >>> main()                               # paramètres par défaut
    >>> main(n_shapes=50, seed=7)            # 50 formes, graine 7
    >>> main(n_shapes=10, min_size=4, max_size=6, seed=0)
    """
    CONFIG.update(
        dict(n_shapes=n_shapes, min_size=min_size, max_size=max_size, seed=seed)
    )

    sep = "=" * 65
    print(f"\n{sep}")
    print(
        f"  {n_shapes} polyominos  ·  taille {min_size}–{max_size}"
        f"  ·  seed={seed}  ·  MAX_BRANCHES={MAX_BRANCHES or '∞'}"
    )
    print(f"{sep}\n")

    shapes = generate_shapes(n=n_shapes, min_size=min_size, max_size=max_size, seed=seed)

    for s in shapes:
        v = get_vertices(s)
        n_p = sum(1 for x in v if x.vtype == "pointe")
        n_c = sum(1 for x in v if x.vtype == "creux")
        print(f"  {s}  |  {n_p} pointe(s)  {n_c} creux")
    print()

    results: dict[int, tuple] = {}
    for hi in (1, 2, 3):
        print(f"  [{_HEURISTIC_LABEL[hi]}]")
        copies = [Shape(s.id, s.cells, s.color) for s in shapes]
        best, logs, branches, trace = greedy_compact_assembly(
            copies, heuristic=hi, verbose=True
        )
        results[hi] = (best, logs, branches, trace)
        print()

    # ── Images de résultats ───────────────────────────────────────────────
    print("  Génération des images de résultats …")
    visualize(shapes, results, save_path=save)

    # ── Métriques de ressemblance ─────────────────────────────────────────
    base = save.replace(".png", "")

    print(f"\n  Métriques de ressemblance (référence : {_HEURISTIC_LABEL[3]}) …")
    metrics = compute_resemblance(results)
    for hi, m in metrics.items():
        print(
            f"    {_HEURISTIC_LABEL[hi]:<35}"
            f"  R_simple={m['simple']:.4f}"
            f"  R_pondéré={m['weighted']:.4f}"
        )
    _save_resemblance(results, metrics, base + "_ressemblance.png")

    print("\n  Évolution temporelle des métriques …")
    _save_resemblance_evolution(results, base + "_ressemblance_evolution.png")

## LANCEMENT

In [ ]:
if __name__ == "__main__":
    main()